In [1]:
import numpy as np 
import pandas as pd


df = pd.read_csv('Student_Depression_Dataset_Updated_Refined_WorkPressure.csv')

df=df[df['Age']<35.0]
df['Age'].value_counts()

df['Academic Pressure'].value_counts() ##remove outliers
df=df[df['Academic Pressure']!=0.0]
df['Academic Pressure'].value_counts()

df=df[df['Study Satisfaction']!=0.0]
df['Study Satisfaction'].value_counts()


df['Degree'].value_counts()

df=df[df['Dietary Habits']!='Others']
df['Dietary Habits'].value_counts()

df=df[df['Sleep Duration']!='Others']
df['Sleep Duration'].value_counts()

df=df.drop(['id'],axis=1)


df['Age'].astype('int')


columns = ['Financial Stress', 'Work/Study Hours', 'Job Satisfaction', 
           'Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction']
# df[columns].hist(alpha=0.75, bins=10)

graduated = ['B.Ed', 'B.Com', 'B.Arch', 'BCA', 'B.Tech', 'BHM', 'BSc', 'B.Pharm', 'BBA', 'BE', 'BA']
post_graduated = ['MSc', 'MCA', 'M.Tech', 'M.Ed', 'M.Com', 'MBBS', 'LLB', 'M.Pharm', 'MD', 'MBA', 'MA', 'PhD', 'LLM', 'MHM', 'ME']
high_school = ['Class 12', 'Others']


# Iterate over the 'Degree' column and update its values
df['Degree']=df['Degree'].replace(graduated,'graduated')
df['Degree']=df['Degree'].replace(post_graduated,'post-graduated')
df['Degree']=df['Degree'].replace(high_school,'high_school')


df.drop([ 'Job Satisfaction','City'], axis=1, inplace=True)

df.loc[df['Gender']=='Male','Gender']=1;
df.loc[df['Gender']=='Female','Gender']=0;

df.loc[df['Sleep Duration']=='Less than 5 hours','Sleep Duration']=3
df.loc[df['Sleep Duration']=='7-8 hours','Sleep Duration']=1
df.loc[df['Sleep Duration']=='5-6 hours','Sleep Duration']=2
df.loc[df['Sleep Duration']=='More than 8 hours','Sleep Duration']=0

df.loc[df['Dietary Habits']=='Unhealthy','Dietary Habits']=2
df.loc[df['Dietary Habits']=='Moderate','Dietary Habits']=1
df.loc[df['Dietary Habits']=='Healthy','Dietary Habits']=0

df.loc[df['Degree']=='graduated','Degree']=1
df.loc[df['Degree']=='post-graduated','Degree']=2
df.loc[df['Degree']=='high_school','Degree']=0

df.loc[df['Have you ever had suicidal thoughts ?']=='Yes','Have you ever had suicidal thoughts ?']=1
df.loc[df['Have you ever had suicidal thoughts ?']=='No','Have you ever had suicidal thoughts ?']=0

df.loc[df['Family History of Mental Illness']=='Yes','Family History of Mental Illness']=1
df.loc[df['Family History of Mental Illness']=='No','Family History of Mental Illness']=0

df.loc[df['Profession'] == 'Student','Profession']=1
df.loc[df['Profession'] != 'Student','Profession']=0

EMOTIONS = ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprised']

# index = 0
for i in EMOTIONS:
    df.loc[df['Emotion'] == i,'Emotion'] = EMOTIONS.index(i)

df = df.dropna()

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

y=df['Stress']
x=df.drop(['Stress'],axis=1)
x.shape


X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)


from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=20)

forest.fit(X_train,y_train)

forest_pred = forest.predict(X_test)
accuracy_score(y_test,forest_pred)*100



86.76734987414599

In [2]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix 
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras import layers, models

DATA_DIR = 'images/'
EMOTIONS = ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprised']
IMG_SIZE = 48

def load_data():
    images = []
    labels = []
    
    for emotion_label, emotion in enumerate(EMOTIONS):
        emotion_dir = os.path.join(DATA_DIR, str(emotion_label))
        if os.path.isdir(emotion_dir):
            for img_file in os.listdir(emotion_dir):
                img_path = os.path.join(emotion_dir, img_file)
                img_array = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img_resized = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                images.append(img_resized)
                labels.append(emotion_label)
        else:
            print(f"Directory not found for emotion {emotion}: {emotion_dir}")
        
    return np.array(images), np.array(labels)

# Load the data
images, labels = load_data()


# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)
# Normalize the pixel values
X_train = X_train / 255.0
X_test = X_test / 255.0

# Reshape data for CNN input
X_train = X_train.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
X_test = X_test.reshape(-1, IMG_SIZE, IMG_SIZE, 1)


datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(X_train)


# In[9]:


model = models.Sequential([
    layers.Conv2D(64, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # Dropout layer to prevent overfitting
    layers.Dense(len(EMOTIONS), activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

history = model.fit(datagen.flow(X_train, y_train, batch_size=32), epochs=100, validation_data=(X_test, y_test))

tr_acc = history.history['accuracy']
tr_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
index_loss = np.argmin(val_loss)
val_lowest = val_loss[index_loss]
index_acc = np.argmax(val_acc)
acc_highest = val_acc[index_acc]

Epochs = [i+1 for i in range(len(tr_acc))]
loss_label = f'best epoch= {str(index_loss + 1)}'
acc_label = f'best epoch= {str(index_acc + 1)}'


test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")



from sklearn.metrics import classification_report

predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=1)

print(classification_report(y_test, predicted_labels, target_names=EMOTIONS))

predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=1)



C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 46, 46, 64)          │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 23, 23, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 21, 21, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 10, 10, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 8, 128)           │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 4, 4, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         262,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 8)                   │           1,032 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 485,384 (1.85 MB)

 Trainable params: 485,384 (1.85 MB)

 Non-trainable params: 0 (0.00 B)

C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 312ms/step - accuracy: 0.1567 - loss: 2.0999 - val_accuracy: 0.3077 - val_loss: 2.0720
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.1878 - loss: 2.0660 - val_accuracy: 0.0769 - val_loss: 2.0600
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.0829 - loss: 2.0834 - val_accuracy: 0.0769 - val_loss: 2.0550
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.1828 - loss: 2.0472 - val_accuracy: 0.1538 - val_loss: 2.0612
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.2637 - loss: 2.0334 - val_accuracy: 0.2308 - val_loss: 2.0561
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.1964 - loss: 1.9985 - val_accuracy: 0.1538 - val_loss: 2.0381
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.1352 - loss: 2.0000 - val_accuracy: 0.0769 - val_loss: 2.0222
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.2063 - loss: 2.0278 - val_accuracy: 0.0769 - v

C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

In [ ]:
import cv2
import time
import tkinter as tk
from tkinter import Label, Button
from PIL import Image, ImageTk
from tkinter import messagebox
import numpy as np

# Load the face detection model
modelFile = "res10_300x300_ssd_iter_140000.caffemodel"
configFile = "deploy.prototxt"
net = cv2.dnn.readNetFromCaffe(configFile, modelFile)



def capture_photo():
    global name
    name = name_entry.get().strip()  # Get user input
    if not name:
        messagebox.showerror("Error", "Please enter a name before capturing.")
        return
    cap = cv2.VideoCapture(0)
    image_saved = False  # Prevent multiple image saves instantly

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]
        blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0), swapRB=False, crop=False)
        net.setInput(blob)
        detections = net.forward()

        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]

            if confidence > 0.5:
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                (x, y, x1, y1) = box.astype("int")

                # Draw rectangle around face
                cv2.rectangle(frame, (x, y), (x1, y1), (0, 255, 0), 3)
                text = f"Confidence: {confidence:.2f}"
                cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                # Capture and save the full frame (not just face)
                if not image_saved:
                    timestamp = int(time.time())  # Unique filename
                    image_path = f"{name}.jpg"
                    cv2.imwrite(image_path, frame)  # Save the full image
                    # print(f"Full photo saved: {image_path}")
                    image_saved = True

                    # Convert captured photo to display in Tkinter
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame_pil = Image.fromarray(frame_rgb)
                    frame_pil = frame_pil.resize((300, 300))  # Resize for display
                    frame_tk = ImageTk.PhotoImage(frame_pil)

                    # Update label with captured photo
                    image_label.config(image=frame_tk)
                    image_label.image = frame_tk  # Prevent garbage collection

        cv2.imshow("Face Detection", frame)

        if cv2.waitKey(1) & 0xFF == ord('q') or image_saved:
            break

    cap.release()
    cv2.destroyAllWindows()


def submit():
    sleep_dur = 0
    if sleep_duration_var.get() == 'More than 8 hours':
        sleep_dur = 1
    elif sleep_duration_var.get() == '7-8 hours':
        sleep_dur = 2
    elif sleep_duration_var.get() == '5-6 hours':
        sleep_dur = 3
    elif sleep_duration_var.get() == 'Less than 5 hours':
        sleep_dur = 4
    diet = 0
    if dietary_habits_var.get() == 'Healthy':
        diet = 1
    elif dietary_habits_var.get() == 'moderate':
        diet = 2
    elif dietary_habits_var.get() == 'Unhealthy':
        diet = 3
    
    degree = 0
    if degree_var.get() == 'High School':
        degree = 1
    elif degree_var.get() == 'Graduated':
        degree = 2
    elif degree_var.get() == 'Post Graduated':
        degree = 3
    
    
    data = {
        "gender": 1 if gender_var.get().capitalize() == "Male" else 0,
        "age": float(age_entry.get()),
        "profession": 1 if profession_var.get().capitalize() == "Student" else 0,
        "AcademicPressure": float(academic_pressure_entry.get()),
        "CGPA": float(cgpa_entry.get()),
        "StudySatisfaction": float(study_satisfaction_entry.get()),
        "studyhours": float(study_hours_entry.get()),
        "SleepDuration":sleep_dur,
        "DietaryHabits": diet,
        "Degree": degree,
        "family_hist": 1 if family_hist_var.get().capitalize() == "Yes" else 0,
        "suicidal_thoughts": 1 if suicidal_thoughts_var.get().capitalize() == "Yes" else 0,
        "work": float(work_pressure_entry.get()),
        "finalcial": float(finalcial_var.get())
    }
    img_array = cv2.imread(f"{name}.jpg", cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img_array, (48, 48))
    img_resized = img_resized.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
    predictions = model.predict(img_resized)
    predicted_labels = np.argmax(predictions, axis=1)

    columns = ["Gender","Age",	"Profession",	"Academic Pressure",	"Work Pressure",	"CGPA",	"Study Satisfaction",	"Sleep Duration",	"Dietary Habits",	"Degree","Have you ever had suicidal thoughts ?",	"Work/Study Hours",	"Financial Stress",	"Family History of Mental Illness",	"Emotion"]
    
    d =np.array([data['gender'],data['age'],data['profession'], data['AcademicPressure'],data['work'], data['CGPA'], data['StudySatisfaction'], data['SleepDuration'], data['DietaryHabits'], data['Degree'], data['suicidal_thoughts'], data['studyhours'],data['finalcial'], data['family_hist'],predicted_labels[0]]).reshape(1,-1)
    d = pd.DataFrame(d,columns=columns)
    forest_pred = forest.predict(d)
    print(forest_pred)
    if forest_pred[0] == 0:
        messagebox.showinfo("Stress Detected ","No Stress")
    else:
        messagebox.showinfo("Stress Detected", "Have Stress")

    
root = tk.Tk()
root.title("Mental Health & Academic Survey")
root.geometry("1000x1000")

image_label = tk.Label(root)
image_label.grid(row=0, column=0, pady=10)

# Name input
tk.Label(root, text="Enter Name:").grid(row=0, column=1, padx=10, pady=5, sticky="e")
name_entry = tk.Entry(root)
name_entry.grid(row=0, column=2, padx=10, pady=5, sticky="w")

# Capture button
capture_button = tk.Button(root, text="Capture Photo", command=capture_photo)
capture_button.grid(row=0, column=4, columnspan=2, pady=10)

# Gender
tk.Label(root, text="Gender:").grid(row=3, column=0, padx=10, pady=5, sticky="e")
gender_var = tk.StringVar()
tk.OptionMenu(root, gender_var, "Male", "Female").grid(row=3, column=1, padx=10, pady=5, sticky="w")

# Age
tk.Label(root, text="Age:" ).grid(row=3, column=2, padx=10, pady=5, sticky="e")
age_entry = tk.Entry(root )
age_entry.grid(row=3, column=3, padx=10, pady=5, sticky="w")

# Profession
tk.Label(root, text="Profession:" ).grid(row=4, column=0, padx=10, pady=5, sticky="e")
profession_var = tk.StringVar()
tk.OptionMenu(root, profession_var, "Student", "Others").grid(row=4, column=1, padx=10, pady=5, sticky="w")

# Academic Pressure
tk.Label(root, text="Academic Pressure (1-5):" ).grid(row=4, column=2, padx=10, pady=5, sticky="e")
academic_pressure_entry = tk.Entry(root )
academic_pressure_entry.grid(row=4, column=3, padx=10, pady=5, sticky="w")

tk.Label(root, text="Work Pressure (1-10):" ).grid(row=5, column=0, padx=10, pady=5, sticky="e")
work_pressure_entry = tk.Entry(root )
work_pressure_entry.grid(row=5, column=1, padx=10, pady=5, sticky="w")
# CGPA
tk.Label(root, text="CGPA:" ).grid(row=5, column=2, padx=10, pady=5, sticky="e")
cgpa_entry = tk.Entry(root )
cgpa_entry.grid(row=5, column=3, padx=10, pady=5, sticky="w")

# Study Satisfaction
tk.Label(root, text="Study Satisfaction (1-5):" ).grid(row=6, column=0, padx=10, pady=5, sticky="e")
study_satisfaction_entry = tk.Entry(root )
study_satisfaction_entry.grid(row=6, column=1, padx=10, pady=5, sticky="w")

tk.Label(root, text="Finalcial Stress (1-5):" ).grid(row=6, column=2, padx=10, pady=5, sticky="e")
finalcial_var = tk.Entry(root )
finalcial_var.grid(row=6, column=3, padx=10, pady=5, sticky="w")

# Study Hours
tk.Label(root, text="Study Hours:" ).grid(row=7, column=0, padx=10, pady=5, sticky="e")
study_hours_entry = tk.Entry(root )
study_hours_entry.grid(row=7, column=1, padx=10, pady=5, sticky="w")

# Sleep Duration
tk.Label(root, text="Sleep Duration:" ).grid(row=7, column=2, padx=10, pady=5, sticky="e")
sleep_duration_var = tk.StringVar()
tk.OptionMenu(root, sleep_duration_var, "More than 8 hours", "7-8 hours", "5-6 hours", "Less than 5 hours").grid(row=7, column=3, padx=10, pady=5, sticky="w")

# Dietary Habits
tk.Label(root, text="Dietary Habits:" ).grid(row=8, column=0, padx=10, pady=5, sticky="e")
dietary_habits_var = tk.StringVar()
tk.OptionMenu(root, dietary_habits_var, "Healthy", "Moderate", "Unhealthy").grid(row=8, column=1, padx=10, pady=5, sticky="w")

# Degree
tk.Label(root, text="Degree:" ).grid(row=8, column=2, padx=10, pady=5, sticky="e")
degree_var = tk.StringVar()
tk.OptionMenu(root, degree_var, "High School", "Graduated", "Post Graduated").grid(row=8, column=3, padx=10, pady=5, sticky="w")

# Family History of Mental Illness
tk.Label(root, text="Family History of Mental Illness:" ).grid(row=9, column=0, padx=10, pady=5, sticky="e")
family_hist_var = tk.StringVar()
tk.OptionMenu(root, family_hist_var, "Yes", "No").grid(row=9, column=1, padx=10, pady=5, sticky="w")

# Suicidal Thoughts
tk.Label(root, text="Suicidal Thoughts:" ).grid(row=9, column=2, padx=10, pady=5, sticky="e")
suicidal_thoughts_var = tk.StringVar()
tk.OptionMenu(root, suicidal_thoughts_var, "Yes", "No").grid(row=9, column=3, padx=10, pady=5, sticky="w")

# Submit button
submit_button = tk.Button(root, text="Calculate", command=submit, font=("Arial", 14))
submit_button.grid(row=10, column=0, columnspan=2, pady=20)

root.mainloop()